# Cross-Task Crystallization Probe: The MOAMS Test

**Question**: Is the "I already know the answer" signal at t=0 a UNIVERSAL subspace,
or is it fragmented per task type?

**Method**:
1. Collect hidden states h at ALL layers, t=0 (last prompt token) across diverse tasks
2. Label each problem: will the model get it right? (binary, from full generation)
3. Train linear probe (LogReg) on one domain, test on another
4. If cross-task transfer works → Z is a universal "reasoning completion" subspace → **MOAMS**

**Task types**: math (easy+medium+hard), factual recall, translation, logic/reasoning
**Languages**: EN and ZH (where applicable)
**Model**: Qwen2.5-7B (28 layers, d=3584)

## Key transfer tests:
- Train math → test factual (cross-domain)
- Train factual → test math (reverse)
- Train EN → test ZH (cross-language)
- Train all-but-one → test held-out (leave-one-out)
- Train easy → test hard (cross-difficulty)

In [ ]:
# Cell 1: Setup
!pip install -q transformers accelerate torch scikit-learn

import json
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, roc_auc_score
import random
import time
from pathlib import Path

print(f"Torch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}, {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Load model
MODEL_NAME = "Qwen/Qwen2.5-7B"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Loading {MODEL_NAME}...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map="cuda"
)
model.eval()

N_LAYERS = model.config.num_hidden_layers  # 28
D_MODEL = model.config.hidden_size          # 3584
DEVICE = next(model.parameters()).device
print(f"Loaded in {time.time()-t0:.1f}s: {N_LAYERS} layers, d={D_MODEL}")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# Cell 3: Problem bank — ~120 problems across 6 categories
# Need enough per category for meaningful train/test splits

rng = random.Random(SEED)

PROBLEMS = []

# === MATH EASY (30 problems, EN + ZH) ===
for _ in range(10):
    a, b = rng.randint(10, 999), rng.randint(10, 999)
    ans = a + b
    PROBLEMS.append({"en": f"Calculate {a} + {b}.", "zh": f"计算 {a} + {b} 的值。",
                     "answer": str(ans), "category": "math_easy", "label": f"add_{a}_{b}"})
for _ in range(10):
    a, b = rng.randint(2, 99), rng.randint(2, 99)
    ans = a * b
    PROBLEMS.append({"en": f"Calculate {a} × {b}.", "zh": f"计算 {a} × {b} 的值。",
                     "answer": str(ans), "category": "math_easy", "label": f"mul_{a}_{b}"})
for _ in range(10):
    a = rng.randint(50, 9999); b = rng.randint(3, 37)
    ans = a % b
    PROBLEMS.append({"en": f"What is the remainder when {a} is divided by {b}?",
                     "zh": f"{a} 除以 {b} 的余数是多少？",
                     "answer": str(ans), "category": "math_easy", "label": f"mod_{a}_{b}"})

# === MATH HARD (20 problems, EN only) ===
MATH_HARD = [
    {"prompt": "What is the sum of all positive divisors of 28?", "answer": "56"},
    {"prompt": "How many integers between 1 and 100 are divisible by 3 but not by 5?", "answer": "27"},
    {"prompt": "What is the remainder when 2^10 is divided by 7?", "answer": "2"},
    {"prompt": "How many 4-digit palindromes are there?", "answer": "90"},
    {"prompt": "Find the last two digits of 7^100.", "answer": "01"},
    {"prompt": "What is the sum of all positive divisors of 60?", "answer": "168"},
    {"prompt": "How many integers between 1 and 200 are divisible by 7 but not by 3?", "answer": "19"},
    {"prompt": "What is C(12, 4)?", "answer": "495"},
    {"prompt": "What is the remainder when 3^20 is divided by 11?", "answer": "1"},
    {"prompt": "How many 3-digit numbers have all digits distinct?", "answer": "648"},
    {"prompt": "What is the GCD of 252 and 198?", "answer": "18"},
    {"prompt": "Find the number of positive integer divisors of 360.", "answer": "24"},
    {"prompt": "What is C(15, 3)?", "answer": "455"},
    {"prompt": "How many prime numbers are there between 50 and 100?", "answer": "10"},
    {"prompt": "What is the sum of the first 20 positive integers?", "answer": "210"},
    {"prompt": "What is the remainder when 11^5 is divided by 13?", "answer": "7"},
    {"prompt": "How many 5-letter strings can be formed from {A,B,C} with no two adjacent letters the same?", "answer": "48"},
    {"prompt": "What is the LCM of 12, 15, and 20?", "answer": "60"},
    {"prompt": "How many integers from 1 to 1000 are perfect squares?", "answer": "31"},
    {"prompt": "What is the sum of digits of 999 × 999?", "answer": "27"},
]
for p in MATH_HARD:
    PROBLEMS.append({"en": p["prompt"], "answer": p["answer"],
                     "category": "math_hard", "label": p["prompt"][:30]})

# === FACTUAL RECALL (25 problems, EN) ===
FACTUAL = [
    {"prompt": "What is the capital of France?", "answer": "Paris"},
    {"prompt": "What is the chemical symbol for gold?", "answer": "Au"},
    {"prompt": "What planet is closest to the Sun?", "answer": "Mercury"},
    {"prompt": "In what year did World War II end?", "answer": "1945"},
    {"prompt": "What is the largest ocean on Earth?", "answer": "Pacific"},
    {"prompt": "What is the boiling point of water in Celsius?", "answer": "100"},
    {"prompt": "What is the capital of Japan?", "answer": "Tokyo"},
    {"prompt": "What is the chemical symbol for iron?", "answer": "Fe"},
    {"prompt": "How many continents are there?", "answer": "7"},
    {"prompt": "What is the speed of light in km/s (approximate)?", "answer": "300000"},
    {"prompt": "Who wrote Romeo and Juliet?", "answer": "Shakespeare"},
    {"prompt": "What is the capital of Germany?", "answer": "Berlin"},
    {"prompt": "What is the atomic number of carbon?", "answer": "6"},
    {"prompt": "How many sides does a hexagon have?", "answer": "6"},
    {"prompt": "What is the chemical formula for water?", "answer": "H2O"},
    {"prompt": "In what year did humans first land on the Moon?", "answer": "1969"},
    {"prompt": "What is the capital of Australia?", "answer": "Canberra"},
    {"prompt": "What is the freezing point of water in Fahrenheit?", "answer": "32"},
    {"prompt": "What element has the symbol Na?", "answer": "Sodium"},
    {"prompt": "How many bones are in the adult human body?", "answer": "206"},
    {"prompt": "What is the capital of Canada?", "answer": "Ottawa"},
    {"prompt": "What is the chemical symbol for silver?", "answer": "Ag"},
    {"prompt": "How many planets are in the solar system?", "answer": "8"},
    {"prompt": "What is the hardest natural substance?", "answer": "Diamond"},
    {"prompt": "What is the capital of Italy?", "answer": "Rome"},
]
for p in FACTUAL:
    PROBLEMS.append({"en": p["prompt"], "answer": p["answer"],
                     "category": "factual", "label": p["prompt"][:30]})

# === LOGIC/REASONING (20 problems, EN) ===
LOGIC = [
    {"prompt": "If all roses are flowers and all flowers need water, do roses need water? Answer yes or no.", "answer": "yes"},
    {"prompt": "What comes next: 2, 4, 8, 16, ?", "answer": "32"},
    {"prompt": "If today is Wednesday, what day is it 3 days from now?", "answer": "Saturday"},
    {"prompt": "If A > B and B > C, is A > C? Answer yes or no.", "answer": "yes"},
    {"prompt": "What comes next: 1, 1, 2, 3, 5, 8, ?", "answer": "13"},
    {"prompt": "If today is Monday, what day was it 4 days ago?", "answer": "Thursday"},
    {"prompt": "All dogs are animals. Some animals are pets. Can we conclude all dogs are pets? Answer yes or no.", "answer": "no"},
    {"prompt": "What comes next: 3, 6, 9, 12, ?", "answer": "15"},
    {"prompt": "If it takes 5 machines 5 minutes to make 5 widgets, how many minutes for 100 machines to make 100 widgets?", "answer": "5"},
    {"prompt": "If no cats are dogs and all dogs bark, can cats bark? Answer yes, no, or cannot determine.", "answer": "cannot"},
    {"prompt": "What comes next: 1, 4, 9, 16, 25, ?", "answer": "36"},
    {"prompt": "If today is Friday, what day will it be in 10 days?", "answer": "Monday"},
    {"prompt": "All squares are rectangles. All rectangles have 4 sides. Do all squares have 4 sides? Answer yes or no.", "answer": "yes"},
    {"prompt": "What comes next: 100, 81, 64, 49, 36, ?", "answer": "25"},
    {"prompt": "A bat and ball cost $1.10. The bat costs $1.00 more than the ball. How much does the ball cost in cents?", "answer": "5"},
    {"prompt": "If you reverse the word 'stressed', what do you get?", "answer": "desserts"},
    {"prompt": "What comes next: 2, 6, 12, 20, 30, ?", "answer": "42"},
    {"prompt": "If all A are B, and all B are C, are all A also C? Answer yes or no.", "answer": "yes"},
    {"prompt": "A farmer has 17 sheep. All but 9 die. How many are left?", "answer": "9"},
    {"prompt": "What comes next: 1, 3, 6, 10, 15, ?", "answer": "21"},
]
for p in LOGIC:
    PROBLEMS.append({"en": p["prompt"], "answer": p["answer"],
                     "category": "logic", "label": p["prompt"][:30]})

# === TRANSLATION (15 problems) ===
TRANSLATION = [
    {"prompt": "Translate to English: 太阳", "answer": "sun"},
    {"prompt": "Translate to English: 数学", "answer": "math"},
    {"prompt": "Translate to English: 水", "answer": "water"},
    {"prompt": "Translate to English: 学校", "answer": "school"},
    {"prompt": "Translate to English: 朋友", "answer": "friend"},
    {"prompt": "Translate to English: 电脑", "answer": "computer"},
    {"prompt": "Translate to English: 老师", "answer": "teacher"},
    {"prompt": "Translate to English: 音乐", "answer": "music"},
    {"prompt": "Translate to Chinese: computer", "answer": "电脑"},
    {"prompt": "Translate to Chinese: water", "answer": "水"},
    {"prompt": "Translate to Chinese: school", "answer": "学校"},
    {"prompt": "Translate to Chinese: friend", "answer": "朋友"},
    {"prompt": "Translate to Chinese: teacher", "answer": "老师"},
    {"prompt": "Translate to Chinese: music", "answer": "音乐"},
    {"prompt": "Translate to Chinese: sun", "answer": "太阳"},
]
for p in TRANSLATION:
    PROBLEMS.append({"en": p["prompt"], "answer": p["answer"],
                     "category": "translation", "label": p["prompt"][:30]})

# Summary
from collections import Counter
cat_counts = Counter(p["category"] for p in PROBLEMS)
print(f"Total problems: {len(PROBLEMS)}")
for cat, n in sorted(cat_counts.items()):
    print(f"  {cat}: {n}")

In [ ]:
# Cell 4: Collect hidden states + generate to determine correctness
#
# For each problem:
# 1. Forward pass on prompt → capture h at all layers, last token
# 2. Full generation (greedy, max 120 tokens) → check if answer appears
# 3. Store: h (all layers), correctness label, t0_p, category, language

def get_answer_token_ids(tokenizer, answer_str):
    """Get token IDs that could represent this answer."""
    ids = set()
    for prefix in ["", " ", "\n"]:
        for variant in [answer_str, answer_str.lower(), answer_str.upper(), answer_str.capitalize()]:
            toks = tokenizer.encode(prefix + variant, add_special_tokens=False)
            ids.update(toks)
    return list(ids)


def collect_one_problem(model, tokenizer, prompt, answer_str, max_gen=120):
    """Collect hidden states at t=0 (all layers) + generate to check correctness."""
    answer_tids = get_answer_token_ids(tokenizer, answer_str)
    final_ln = model.model.norm
    lm_head = model.lm_head

    input_ids = tokenizer.encode(prompt, add_special_tokens=True)
    input_ids_t = torch.tensor([input_ids], device=DEVICE)

    # Step 1: capture hidden states at last prompt token (t=0)
    layer_hiddens = {}
    def make_capture(l):
        def hook(module, inp, out):
            h = out[0] if isinstance(out, tuple) else out
            layer_hiddens[l] = h[:, -1, :].detach().cpu().float().numpy().squeeze()
        return hook

    handles = [model.model.layers[l].register_forward_hook(make_capture(l))
               for l in range(N_LAYERS)]
    with torch.no_grad():
        outputs = model(input_ids_t)
    for h in handles:
        h.remove()

    # t0 probability of answer at each layer
    t0_p_per_layer = []
    for l in range(N_LAYERS):
        h_l = torch.tensor(layer_hiddens[l], device=DEVICE, dtype=torch.bfloat16).unsqueeze(0).unsqueeze(0)
        h_normed = final_ln(h_l)
        logits_l = lm_head(h_normed).float().squeeze()
        probs = F.softmax(logits_l, dim=-1)
        p_ans = max(probs[tid].item() for tid in answer_tids) if answer_tids else 0.0
        t0_p_per_layer.append(p_ans)

    # Stack hidden states: (N_LAYERS, D_MODEL)
    h_all = np.stack([layer_hiddens[l] for l in range(N_LAYERS)])

    # Step 2: generate and check correctness
    with torch.no_grad():
        gen_out = model.generate(input_ids_t, max_new_tokens=max_gen, do_sample=False)
    gen_tokens = gen_out[0, len(input_ids):]
    gen_text = tokenizer.decode(gen_tokens, skip_special_tokens=True)

    # Check if answer appears in generated text
    correct = answer_str.lower() in gen_text.lower()

    return h_all, correct, t0_p_per_layer, gen_text


# Run collection — two passes:
# Pass 1: EN prompts for all problems
# Pass 2: ZH prompts for math_easy (bilingual)

all_hiddens = []    # list of (N_LAYERS, D_MODEL) arrays
all_correct = []    # bool
all_t0_p = []       # list of (N_LAYERS,) lists
all_meta = []       # {category, label, language, prompt, answer, gen_text}

print(f"Collecting {len(PROBLEMS)} problems (EN pass)...")
t0 = time.time()
for i, prob in enumerate(PROBLEMS):
    prompt = prob["en"]
    h, correct, t0_p, gen = collect_one_problem(model, tokenizer, prompt, prob["answer"])

    all_hiddens.append(h)
    all_correct.append(correct)
    all_t0_p.append(t0_p)
    all_meta.append({
        "idx": len(all_meta), "category": prob["category"], "label": prob.get("label", ""),
        "language": "en", "prompt": prompt[:100], "answer": prob["answer"],
        "correct": correct, "t0_p_final": t0_p[-1], "gen_text": gen[:150]
    })

    if (i+1) % 20 == 0:
        n_corr = sum(all_correct[-20:])
        print(f"  [{i+1}/{len(PROBLEMS)}] last 20: {n_corr}/20 correct, "
              f"{time.time()-t0:.0f}s elapsed")

# Pass 2: ZH prompts for bilingual problems
zh_problems = [p for p in PROBLEMS if "zh" in p]
print(f"\nCollecting {len(zh_problems)} problems (ZH pass)...")
for i, prob in enumerate(zh_problems):
    prompt = prob["zh"]
    h, correct, t0_p, gen = collect_one_problem(model, tokenizer, prompt, prob["answer"])

    all_hiddens.append(h)
    all_correct.append(correct)
    all_t0_p.append(t0_p)
    all_meta.append({
        "idx": len(all_meta), "category": prob["category"], "label": prob.get("label", ""),
        "language": "zh", "prompt": prompt[:100], "answer": prob["answer"],
        "correct": correct, "t0_p_final": t0_p[-1], "gen_text": gen[:150]
    })

    if (i+1) % 10 == 0:
        n_corr = sum(all_correct[-10:])
        print(f"  [{i+1}/{len(zh_problems)}] last 10: {n_corr}/10 correct")

# Stack
H = np.stack(all_hiddens)       # (N_problems, N_LAYERS, D_MODEL)
Y = np.array(all_correct, dtype=int)  # (N_problems,)
T0P = np.array(all_t0_p)       # (N_problems, N_LAYERS)

print(f"\n=== COLLECTION COMPLETE ===")
print(f"H shape: {H.shape}")
print(f"Total: {len(Y)} samples, {Y.sum()} correct ({Y.mean()*100:.1f}%)")
for cat in sorted(set(m['category'] for m in all_meta)):
    mask = [m['category'] == cat for m in all_meta]
    n = sum(mask)
    n_c = sum(Y[i] for i in range(len(Y)) if mask[i])
    print(f"  {cat}: {n} samples, {n_c}/{n} correct ({n_c/n*100:.1f}%)")
for lang in ["en", "zh"]:
    mask = [m['language'] == lang for m in all_meta]
    n = sum(mask)
    n_c = sum(Y[i] for i in range(len(Y)) if mask[i])
    print(f"  {lang}: {n} samples, {n_c}/{n} correct ({n_c/n*100:.1f}%)")

print(f"\nElapsed: {time.time()-t0:.0f}s")

In [ ]:
# Cell 5: Save raw data (in case Colab disconnects)

np.savez_compressed("crosstask_probe_hiddens.npz",
                    H=H, Y=Y, T0P=T0P)

with open("crosstask_probe_meta.json", "w") as f:
    json.dump(all_meta, f, indent=2, ensure_ascii=False)

print(f"Saved hiddens: {Path('crosstask_probe_hiddens.npz').stat().st_size / 1e6:.1f} MB")
print(f"Saved meta: {Path('crosstask_probe_meta.json').stat().st_size / 1e3:.1f} KB")

# Download if Colab
try:
    from google.colab import files
    files.download("crosstask_probe_hiddens.npz")
    files.download("crosstask_probe_meta.json")
    print("Downloads triggered.")
except ImportError:
    print("Not on Colab.")

In [ ]:
# Cell 6: Helper — train and evaluate linear probe

def train_probe(X_train, y_train, X_test, y_test, pca_dim=64):
    """
    Train PCA + LogReg probe.
    Returns accuracy, AUC (if possible), and the trained model.
    """
    # Check for degenerate cases
    if len(set(y_train)) < 2:
        return {"acc": float(np.mean(y_test == y_train[0])), "auc": None,
                "note": "single-class train", "n_train": len(y_train), "n_test": len(y_test)}
    if len(y_test) == 0:
        return {"acc": None, "auc": None, "note": "empty test",
                "n_train": len(y_train), "n_test": 0}

    # PCA
    effective_dim = min(pca_dim, X_train.shape[0]-1, X_train.shape[1])
    if effective_dim < 2:
        return {"acc": None, "auc": None, "note": "too few samples for PCA",
                "n_train": len(y_train), "n_test": len(y_test)}

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_train)
    X_te_s = scaler.transform(X_test)

    pca = PCA(n_components=effective_dim)
    X_tr_p = pca.fit_transform(X_tr_s)
    X_te_p = pca.transform(X_te_s)

    # LogReg
    clf = LogisticRegression(max_iter=2000, C=1.0, random_state=SEED)
    clf.fit(X_tr_p, y_train)

    y_pred = clf.predict(X_te_p)
    acc = accuracy_score(y_test, y_pred)

    # AUC if binary and both classes present in test
    auc = None
    if len(set(y_test)) == 2:
        try:
            y_prob = clf.predict_proba(X_te_p)[:, 1]
            auc = roc_auc_score(y_test, y_prob)
        except:
            pass

    return {"acc": acc, "auc": auc, "n_train": len(y_train), "n_test": len(y_test),
            "train_balance": f"{y_train.mean():.2f}", "test_balance": f"{y_test.mean():.2f}",
            "pca_var_explained": float(pca.explained_variance_ratio_.sum())}


print("Probe helper loaded.")

In [ ]:
# Cell 7: SECTION A — Within-domain probe (baseline)
# For each category: 70/30 split, train and test within same domain.
# This establishes the ceiling — how well can correctness be predicted
# from t=0 hidden states within a single task type?

categories = sorted(set(m['category'] for m in all_meta))
languages = sorted(set(m['language'] for m in all_meta))

# We'll test at every layer to see the profile
PROBE_LAYERS = list(range(N_LAYERS))

within_results = {}
print("=== SECTION A: Within-Domain Probe (baseline) ===")
print(f"Testing at final layer (L{N_LAYERS-1}) first, then all layers.\n")

for cat in categories:
    idxs = [i for i, m in enumerate(all_meta) if m['category'] == cat]
    n = len(idxs)
    y_cat = Y[idxs]

    # Skip if too few or single-class
    if n < 10 or len(set(y_cat)) < 2:
        print(f"  {cat}: SKIP (n={n}, classes={len(set(y_cat))}")
        within_results[cat] = {"status": "skipped", "n": n}
        continue

    # 70/30 split
    rng_np = np.random.RandomState(SEED)
    perm = rng_np.permutation(n)
    n_train = int(0.7 * n)
    train_idx = [idxs[i] for i in perm[:n_train]]
    test_idx = [idxs[i] for i in perm[n_train:]]

    # Final layer
    X_tr = H[train_idx, N_LAYERS-1, :]
    X_te = H[test_idx, N_LAYERS-1, :]
    y_tr = Y[train_idx]
    y_te = Y[test_idx]

    result = train_probe(X_tr, y_tr, X_te, y_te)
    chance = max(y_te.mean(), 1 - y_te.mean())
    print(f"  {cat:>12}: acc={result['acc']:.3f} (chance={chance:.3f}) "
          f"auc={result.get('auc', 'N/A')} n_train={result['n_train']} n_test={result['n_test']}")

    # All layers profile
    layer_accs = []
    for l in PROBE_LAYERS:
        X_tr_l = H[train_idx, l, :]
        X_te_l = H[test_idx, l, :]
        r = train_probe(X_tr_l, y_tr, X_te_l, y_te)
        layer_accs.append(r['acc'] if r['acc'] is not None else 0.0)

    within_results[cat] = {
        "final_layer": result,
        "layer_profile": layer_accs,
        "chance": float(chance),
        "n": n
    }

print("\nDone.")

In [ ]:
# Cell 8: SECTION B — Cross-Task Transfer (THE MOAMS TEST)
# Train on one category, test on another.
# If a probe trained on math can predict factual recall correctness → MOAMS.

cross_task_results = {}

print("=== SECTION B: Cross-Task Transfer ===")
print("Train on source → test on target. Using final layer.\n")

for source_cat in categories:
    source_idx = [i for i, m in enumerate(all_meta) if m['category'] == source_cat]
    y_source = Y[source_idx]
    if len(set(y_source)) < 2:
        print(f"  {source_cat} → *: SKIP (single-class source)")
        continue

    for target_cat in categories:
        if target_cat == source_cat:
            continue
        target_idx = [i for i, m in enumerate(all_meta) if m['category'] == target_cat]
        y_target = Y[target_idx]

        if len(target_idx) < 5:
            continue

        X_tr = H[source_idx, N_LAYERS-1, :]
        X_te = H[target_idx, N_LAYERS-1, :]

        result = train_probe(X_tr, y_source, X_te, y_target)
        chance = max(y_target.mean(), 1 - y_target.mean())

        key = f"{source_cat} → {target_cat}"
        cross_task_results[key] = {"result": result, "chance": float(chance)}

        marker = "***" if result.get('acc', 0) and result['acc'] > chance + 0.10 else ""
        print(f"  {key:>40}: acc={result.get('acc', 'N/A'):.3f} "
              f"(chance={chance:.3f}) {marker}")

print("\n*** = above chance + 10%  (evidence for universal subspace)")

In [ ]:
# Cell 9: SECTION C — Cross-Language Transfer
# Train on EN math, test on ZH math (same problems, different language)
# This tests whether the "knows before speaks" signal is language-agnostic.

print("=== SECTION C: Cross-Language Transfer ===")

# EN math indices
en_math_idx = [i for i, m in enumerate(all_meta)
               if m['category'] == 'math_easy' and m['language'] == 'en']
# ZH math indices
zh_math_idx = [i for i, m in enumerate(all_meta)
               if m['category'] == 'math_easy' and m['language'] == 'zh']

cross_lang_results = {}

if en_math_idx and zh_math_idx:
    y_en = Y[en_math_idx]
    y_zh = Y[zh_math_idx]

    print(f"EN math: {len(en_math_idx)} samples, {y_en.sum()}/{len(y_en)} correct")
    print(f"ZH math: {len(zh_math_idx)} samples, {y_zh.sum()}/{len(y_zh)} correct")

    # Test at every layer
    print("\nEN → ZH transfer by layer:")
    en2zh_by_layer = []
    for l in PROBE_LAYERS:
        X_en = H[en_math_idx, l, :]
        X_zh = H[zh_math_idx, l, :]
        r = train_probe(X_en, y_en, X_zh, y_zh)
        acc = r['acc'] if r['acc'] is not None else 0.0
        en2zh_by_layer.append(acc)

    # Print key layers
    for l in [0, 4, 7, 10, 14, 18, 22, 26, N_LAYERS-1]:
        if l < N_LAYERS:
            print(f"  L{l:>2}: {en2zh_by_layer[l]:.3f}")

    # Reverse: ZH → EN
    print("\nZH → EN transfer by layer:")
    zh2en_by_layer = []
    for l in PROBE_LAYERS:
        X_en = H[en_math_idx, l, :]
        X_zh = H[zh_math_idx, l, :]
        r = train_probe(X_zh, y_zh, X_en, y_en)
        acc = r['acc'] if r['acc'] is not None else 0.0
        zh2en_by_layer.append(acc)

    for l in [0, 4, 7, 10, 14, 18, 22, 26, N_LAYERS-1]:
        if l < N_LAYERS:
            print(f"  L{l:>2}: {zh2en_by_layer[l]:.3f}")

    cross_lang_results = {
        "en2zh_by_layer": en2zh_by_layer,
        "zh2en_by_layer": zh2en_by_layer,
        "en_balance": float(y_en.mean()),
        "zh_balance": float(y_zh.mean()),
    }
else:
    print("Missing EN or ZH math data.")

In [ ]:
# Cell 10: SECTION D — Leave-One-Out Category Transfer
# Train on ALL other categories, test on held-out category.
# If this works broadly → the "knows" signal is truly universal.

loo_results = {}

print("=== SECTION D: Leave-One-Out Category Transfer ===")
print("Train on everything EXCEPT target category.\n")

for held_out in categories:
    test_idx = [i for i, m in enumerate(all_meta) if m['category'] == held_out]
    train_idx = [i for i, m in enumerate(all_meta) if m['category'] != held_out]

    y_tr = Y[train_idx]
    y_te = Y[test_idx]

    if len(set(y_tr)) < 2 or len(test_idx) < 3:
        print(f"  held_out={held_out}: SKIP")
        continue

    # Final layer
    X_tr = H[train_idx, N_LAYERS-1, :]
    X_te = H[test_idx, N_LAYERS-1, :]

    result = train_probe(X_tr, y_tr, X_te, y_te)
    chance = max(y_te.mean(), 1 - y_te.mean())

    print(f"  held_out={held_out:>12}: acc={result.get('acc', 'N/A'):.3f} "
          f"(chance={chance:.3f}) auc={result.get('auc', 'N/A')} "
          f"n_train={result['n_train']} n_test={result['n_test']}")

    # Layer profile
    layer_accs = []
    for l in PROBE_LAYERS:
        r = train_probe(H[train_idx, l, :], y_tr, H[test_idx, l, :], y_te)
        layer_accs.append(r['acc'] if r['acc'] is not None else 0.0)

    loo_results[held_out] = {
        "final_layer": result,
        "layer_profile": layer_accs,
        "chance": float(chance),
    }

In [ ]:
# Cell 11: SECTION E — t0_p as a predictor (no probe needed)
# Does t0_p (answer probability at t=0, final layer) directly predict correctness?
# This is the simplest possible "crystallization detector."

print("=== SECTION E: t0_p as Direct Predictor ===")

t0_p_final = T0P[:, -1]  # (N_problems,)

# Overall correlation
print(f"\nOverall: Pearson r(t0_p, correct) = {np.corrcoef(t0_p_final, Y)[0,1]:.3f}")

# By category
print("\nBy category:")
for cat in categories:
    idx = [i for i, m in enumerate(all_meta) if m['category'] == cat]
    t0 = t0_p_final[idx]
    y = Y[idx]
    if len(set(y)) < 2:
        print(f"  {cat:>12}: r=N/A (single class), mean_t0_p={t0.mean():.4f}")
        continue
    r = np.corrcoef(t0, y)[0, 1]

    # Threshold-based accuracy: if t0_p > threshold, predict correct
    best_acc = 0
    best_thresh = 0
    for thresh in np.arange(0.001, 0.5, 0.005):
        pred = (t0 > thresh).astype(int)
        acc = accuracy_score(y, pred)
        if acc > best_acc:
            best_acc = acc
            best_thresh = thresh

    print(f"  {cat:>12}: r={r:.3f}, best_thresh_acc={best_acc:.3f}@{best_thresh:.3f}, "
          f"mean_t0p_correct={t0[y==1].mean():.4f} vs incorrect={t0[y==0].mean():.4f}")

# By language
print("\nBy language:")
for lang in ["en", "zh"]:
    idx = [i for i, m in enumerate(all_meta) if m['language'] == lang]
    t0 = t0_p_final[idx]
    y = Y[idx]
    if len(set(y)) < 2:
        print(f"  {lang}: r=N/A, mean_t0_p={t0.mean():.4f}")
        continue
    r = np.corrcoef(t0, y)[0, 1]
    print(f"  {lang}: r={r:.3f}, mean_t0p_correct={t0[y==1].mean():.4f} vs incorrect={t0[y==0].mean():.4f}")

# AUC using t0_p directly as score
if len(set(Y)) == 2:
    auc_direct = roc_auc_score(Y, t0_p_final)
    print(f"\nGlobal AUC (t0_p as classifier): {auc_direct:.3f}")

In [ ]:
# Cell 12: SECTION F — Dimensionality of the "knows" subspace
# PCA on the correct-vs-incorrect difference.
# How many dimensions does the "knows" signal live in?

print("=== SECTION F: Dimensionality of Knows Signal ===")

# Use final layer hidden states
H_final = H[:, N_LAYERS-1, :]  # (N, D_MODEL)

# Split by correctness
correct_idx = np.where(Y == 1)[0]
incorrect_idx = np.where(Y == 0)[0]

if len(correct_idx) > 5 and len(incorrect_idx) > 5:
    # Mean difference direction
    mean_correct = H_final[correct_idx].mean(axis=0)
    mean_incorrect = H_final[incorrect_idx].mean(axis=0)
    diff_dir = mean_correct - mean_incorrect
    diff_norm = np.linalg.norm(diff_dir)
    diff_dir_unit = diff_dir / (diff_norm + 1e-8)
    print(f"Mean-diff direction norm: {diff_norm:.2f}")

    # Project all points onto this direction
    proj = H_final @ diff_dir_unit
    r_1d = np.corrcoef(proj, Y)[0, 1]
    print(f"1D projection correlation with correctness: r={r_1d:.3f}")

    # PCA on the concatenated correct+incorrect, then test separability
    scaler = StandardScaler()
    H_scaled = scaler.fit_transform(H_final)
    pca = PCA(n_components=min(50, H_scaled.shape[0]-1))
    H_pca = pca.fit_transform(H_scaled)

    print(f"\nPCA variance explained (cumulative):")
    cum_var = np.cumsum(pca.explained_variance_ratio_)
    for k in [1, 2, 3, 5, 10, 20, 30, 50]:
        if k <= len(cum_var):
            print(f"  PC 1-{k}: {cum_var[k-1]*100:.1f}%")

    # Probe accuracy as function of PCA dims
    print("\nProbe accuracy vs PCA dimensionality (70/30 split):")
    rng_np = np.random.RandomState(SEED)
    perm = rng_np.permutation(len(Y))
    n_train = int(0.7 * len(Y))
    tr_i = perm[:n_train]
    te_i = perm[n_train:]

    for k in [1, 2, 3, 5, 10, 20, 30, 50]:
        if k > H_pca.shape[1]:
            break
        clf = LogisticRegression(max_iter=2000, C=1.0, random_state=SEED)
        clf.fit(H_pca[tr_i, :k], Y[tr_i])
        pred = clf.predict(H_pca[te_i, :k])
        acc = accuracy_score(Y[te_i], pred)
        chance = max(Y[te_i].mean(), 1 - Y[te_i].mean())
        print(f"  PCA-{k:>2}: acc={acc:.3f} (chance={chance:.3f}) "
              f"delta={acc-chance:+.3f}")

    dim_results = {
        "diff_norm": float(diff_norm),
        "proj_corr_1d": float(r_1d),
        "pca_cumvar": cum_var.tolist(),
    }
else:
    print("Not enough samples in both classes.")
    dim_results = {}

In [ ]:
# Cell 13: Save all probe results

output = {
    "experiment": "crosstask_crystallization_probe",
    "model": MODEL_NAME,
    "n_layers": N_LAYERS,
    "d_model": D_MODEL,
    "n_problems_total": len(Y),
    "n_correct": int(Y.sum()),
    "overall_accuracy": float(Y.mean()),
    "sections": {
        "A_within_domain": within_results,
        "B_cross_task": cross_task_results,
        "C_cross_language": cross_lang_results,
        "D_leave_one_out": loo_results,
        "E_t0p_predictor": {
            "global_corr": float(np.corrcoef(t0_p_final, Y)[0,1]) if len(set(Y)) == 2 else None,
            "global_auc": float(roc_auc_score(Y, t0_p_final)) if len(set(Y)) == 2 else None,
        },
        "F_dimensionality": dim_results,
    },
    "meta": all_meta,
}

output_path = "crosstask_probe_results.json"
with open(output_path, "w") as f:
    json.dump(output, f, indent=2, ensure_ascii=False, default=str)

print(f"Saved to {output_path}")
print(f"File size: {Path(output_path).stat().st_size / 1024:.1f} KB")

try:
    from google.colab import files
    files.download(output_path)
    print("Download triggered.")
except ImportError:
    print("Not on Colab.")

In [ ]:
# Cell 14: Summary — THE VERDICT

print("=" * 70)
print("CROSS-TASK CRYSTALLIZATION PROBE: MOAMS TEST RESULTS")
print("=" * 70)

print("\n--- A: Within-Domain Baselines (how predictable is correctness?) ---")
for cat in categories:
    if cat in within_results and 'final_layer' in within_results[cat]:
        r = within_results[cat]['final_layer']
        c = within_results[cat]['chance']
        print(f"  {cat:>12}: acc={r.get('acc','N/A'):.3f} (chance={c:.3f})")

print("\n--- B: Cross-Task Transfer (does \"knows\" generalize across tasks?) ---")
for key, val in sorted(cross_task_results.items()):
    r = val['result']
    c = val['chance']
    verdict = "TRANSFER" if r.get('acc',0) and r['acc'] > c + 0.05 else "NO TRANSFER"
    print(f"  {key:>40}: {r.get('acc','N/A'):.3f} ({verdict})")

print("\n--- C: Cross-Language Transfer (EN ↔ ZH) ---")
if cross_lang_results:
    print(f"  EN→ZH at L{N_LAYERS-1}: {cross_lang_results['en2zh_by_layer'][-1]:.3f}")
    print(f"  ZH→EN at L{N_LAYERS-1}: {cross_lang_results['zh2en_by_layer'][-1]:.3f}")

print("\n--- D: Leave-One-Out (train everything else, test held-out) ---")
for cat, val in sorted(loo_results.items()):
    r = val['final_layer']
    c = val['chance']
    verdict = "TRANSFER" if r.get('acc',0) and r['acc'] > c + 0.05 else "NO TRANSFER"
    print(f"  held_out={cat:>12}: {r.get('acc','N/A'):.3f} (chance={c:.3f}) → {verdict}")

print("\n--- E: t0_p as Direct Predictor ---")
if len(set(Y)) == 2:
    print(f"  Global AUC: {roc_auc_score(Y, t0_p_final):.3f}")
    print(f"  Global correlation: {np.corrcoef(t0_p_final, Y)[0,1]:.3f}")

print("\n--- F: Dimensionality ---")
if dim_results:
    print(f"  1D projection r: {dim_results.get('proj_corr_1d', 'N/A')}")

print("\n" + "=" * 70)
# Count transfers
n_transfer = sum(1 for v in cross_task_results.values()
                 if v['result'].get('acc', 0) and v['result']['acc'] > v['chance'] + 0.05)
n_total = len(cross_task_results)
n_loo = sum(1 for v in loo_results.values()
            if v['final_layer'].get('acc', 0) and v['final_layer']['acc'] > v['chance'] + 0.05)
n_loo_total = len(loo_results)

print(f"Cross-task transfer: {n_transfer}/{n_total} pairs show transfer")
print(f"Leave-one-out: {n_loo}/{n_loo_total} categories show transfer")

if n_transfer > n_total * 0.5 and n_loo > n_loo_total * 0.5:
    print("\n→ VERDICT: EVIDENCE FOR UNIVERSAL 'KNOWS' SUBSPACE (MOAMS CANDIDATE)")
elif n_transfer > 0 or n_loo > 0:
    print("\n→ VERDICT: PARTIAL TRANSFER — Z exists but may be fragmented")
else:
    print("\n→ VERDICT: NO CROSS-TASK TRANSFER — Z is domain-specific")
print("=" * 70)